# Pandas: from building results to error rates

**About 30 minutes.** Use pandas to read a CSV, inspect and filter a table, create columns, and compare PWTT errors across vegetation classes.

Run cells in order. __Replace each `...` in the exercises__, then check the expected output below. Keep `building_results_practice.csv` in the same folder as this notebook. In Colab, upload the CSV to the session's Files panel first.

The CSV contains **13 fictional buildings**. `damage_label` is the reference label: `1` means damaged and `0` means undamaged. Tree cover is a percentage. One building has no PWTT score. The threshold of 3.3 is for practice, not a validated wildfire threshold.

This notebook uses pandas and matplotlib. If an import fails, run `%pip install pandas matplotlib` in a new code cell, then rerun the import.

## 1. Read a table

`import pandas as pd` gives pandas the short name `pd`. A **DataFrame** is a table with named columns and an index that labels the rows. A single column is a **Series**.

`pd.read_csv()` reads a CSV file. The path below is relative to your notebook's working directory. `.head()` shows the first five rows; a table on the last line of a cell displays automatically.

In [ ]:
import pandas as pd

buildings = pd.read_csv("building_results_practice.csv")
buildings.head()


The shape attribute holds the shape of the dataframe, it is returned as a tuple.  You can access the entires in a tuple by index. Unlike a method such as `.head()`, `.shape` has no parentheses.  This is because a method is a built in function, while an attribute is, well ... an attribute of the object.

In [ ]:
print(buildings.shape)
print(buildings.shape[0]) # number of rows
print(buildings.shape[1]) # number of columns

**Your turn.** Use `.shape` to find the number of rows and columns. 

In [ ]:

print(buildings.shape[...])  # The number of rows
print(buildings.shape[...])  # The number of columns

**Expected output after completing the code:**

```text
13
6
```

## 2. Inspect before calculating

`.info()` reports column types and non-missing counts. `.describe()` summarizes numeric columns. Look for missing data and unexpected ranges before doing statistics.

`buildings["PWTT_T_mean"]` selects one column. Use a **list of names** to select several columns: `buildings[["building_id", "PWTT_T_mean"]]`. The outside brackets select from the table; the inside brackets create the list.

In [ ]:
buildings.info()
buildings[["pre_fire_tree_cover_30m", "PWTT_T_mean"]].describe()


**Your turn.** Select the building ID and PWTT score columns. Print their column names using `.columns.tolist()`.

In [ ]:
scores = buildings[[..., ...]]
print(scores.columns.tolist())


**Expected output after completing the code:**

```text
['building_id', 'PWTT_T_mean']
```

## 3. Filter rows

A comparison applied to a column creates one boolean per row. Put that boolean Series inside brackets to select the matching rows.

For pandas conditions, use `&` for AND and `|` for OR, with **parentheses around each comparison**. Use `~` for NOT. Python's `and` and `or` do not combine whole Series.

In [ ]:
high_cover = buildings[buildings["pre_fire_tree_cover_30m"] >= 50]
print(high_cover["building_id"].tolist())

# .loc selects rows and columns together: .loc[row_condition, column_names]
buildings.loc[buildings["damage_label"] == 1, ["building_id", "PWTT_T_mean"]]


**Your turn.** Select buildings that have at least 50% tree cover AND are labeled undamaged. Print their IDs.

In [ ]:
selected = buildings[
    (buildings["pre_fire_tree_cover_30m"] >= ...)
    & (buildings["damage_label"] == ...)
]
print(selected["building_id"].tolist())


**Expected output after completing the code:**

```text
['B007', 'B008', 'B009', 'B013']
```

## 4. Handle missing data deliberately

The blank score for B013 is read as `NaN`, a missing numeric value. `.isna()` identifies missing entries; `.sum()` counts the `True` values.

We cannot assess a threshold-based prediction without a score, so exclude rows missing a score or reference label from this exercise's accuracy calculations. Do not replace missing scores with zero: that would turn “unknown” into a no-damage prediction. In real results, report exclusions and check whether they concentrate in particular groups.

`.dropna(subset=[...])` checks only the named columns. `.copy()` makes a separate table that we can add columns to.

In [ ]:
print(buildings["PWTT_T_mean"].isna().sum())
buildings.loc[buildings["PWTT_T_mean"].isna(), ["building_id", "PWTT_T_mean"]]


**Your turn.** Create `valid` by excluding rows missing either the PWTT score or the damage label. Print the remaining and excluded row counts.

In [ ]:
valid = buildings.dropna(subset=[..., ...]).copy()
print(len(valid))
print(len(buildings) - len(valid))


**Expected output after completing the code:**

```text
12
1
```

## 5. Create predictions and error columns

Pandas applies an operation to a whole column, so no Python loop is needed here. Assign a new column using `table["new_name"] = ...`.

A **false positive** is an undamaged building flagged as damaged. A **false negative** is a damaged building that is missed. The reference labels—not the predictions—tell us which buildings are actually damaged.

In [ ]:
threshold = 3.3
valid["prediction"] = valid["PWTT_T_mean"] > threshold
valid["reference_damaged"] = valid["damage_label"] == 1
valid["correct"] = valid["prediction"] == valid["reference_damaged"]
valid["false_positive"] = valid["prediction"] & ~valid["reference_damaged"]

print(valid["false_positive"].sum())


**Your turn.** Create `false_negative` using NOT predicted damaged AND reference damaged. Then count false negatives and correct predictions.

In [ ]:
valid["false_negative"] = ...
print(valid["false_negative"].sum())
print(valid["correct"].sum())


**Expected output after completing the code:**

```text
2
7
```

## 6. Calculate rates with the right denominator

For a boolean column, `.mean()` gives the fraction of `True` values: Python treats `True` as 1 and `False` as 0. Use `round(value, 3)` to display three decimal places.

- **Accuracy:** correct predictions / all evaluated buildings.
- **False-positive rate:** false positives / reference-undamaged buildings.
- **False-negative rate:** false negatives / reference-damaged buildings.

False-positive rate is **not** one minus precision: precision uses predicted-damaged buildings as its denominator. Recall equals one minus the false-negative rate.

In [ ]:
accuracy = valid["correct"].mean()
undamaged = valid[~valid["reference_damaged"]].copy()
false_positive_rate = undamaged["false_positive"].mean()

print(round(accuracy, 3))
print(round(false_positive_rate, 3))


**Your turn.** Select only reference-damaged buildings, then calculate the false-negative rate and recall.

In [ ]:
damaged = valid[...].copy()
false_negative_rate = damaged[...].mean()
recall = ...
print(round(false_negative_rate, 3))
print(round(recall, 3))


**Expected output after completing the code:**

```text
0.333
0.667
```

## 7. Compare vegetation classes with groupby

`.groupby()` splits a table into groups so you can summarize each one. First count evaluated buildings by vegetation class. Then use **only the reference-undamaged buildings** to compare false-positive rates.

`.agg()` can calculate several summaries at once. Below, each output column is named on the left; the tuple on the right gives the input column and operation. `size` counts rows, `sum` counts errors in a boolean column, and `mean` calculates the error rate.

In [ ]:
print(valid.groupby("vegetation_class").size())

summary = undamaged.groupby("vegetation_class").agg(
    n_undamaged=("false_positive", "size"),
    false_positives=("false_positive", "sum"),
    false_positive_rate=("false_positive", "mean"),
)
summary.round(3)


**Your turn.** Retrieve each group’s false-positive rate with `.loc[row_label, column_name]`. Print the low-vegetation rate first, then the high-vegetation rate.

In [ ]:
print(round(summary.loc[..., "false_positive_rate"], 3))
print(round(summary.loc[..., "false_positive_rate"], 3))


**Expected output after completing the code:**

```text
0.333
0.667
```

**Interpretation:** In this practice table, one of three undamaged buildings in the low-vegetation group was flagged, compared with two of three in the high-vegetation group. These tiny, invented samples do not establish a vegetation effect. All low-vegetation buildings also come from Fire_A and all high-vegetation buildings from Fire_B, so vegetation and fire cannot be separated here.

In real data, report group counts, exclusions, and uncertainty alongside rates. A group with no reference-undamaged buildings has an undefined false-positive rate, not a rate of zero.

## 8. Plot and save the summary

Pandas uses matplotlib for this bar chart. `.reindex(["low", "high"])` puts the classes in the desired order. Set the y-axis to 0–1 because these are proportions.

In [ ]:
import matplotlib.pyplot as plt

summary.reindex(["low", "high"])["false_positive_rate"].plot.bar(
    color=["steelblue", "darkorange"],
    ylim=(0, 1),
    rot=0,
    xlabel="Vegetation class",
    ylabel="False-positive rate",
    title="Practice data: false positives among undamaged buildings",
)
plt.tight_layout()
plt.show()


**Expected plot:** Two bars: low vegetation at about 0.33 and high vegetation at about 0.67.

`.reset_index()` turns the vegetation labels from index labels into a regular column. `.to_csv(..., index=False)` saves the table without adding an extra row-number column. Rerunning the save cell replaces the file with the same name.

**Your turn.** Save the summary to `vegetation_error_summary.csv`. Fill in the filename, then run the cell.

In [ ]:
summary.reset_index().to_csv(..., index=False)
print("Saved vegetation_error_summary.csv")


**Expected output after completing the code:**

```text
Saved vegetation_error_summary.csv
```

The saved CSV should contain `vegetation_class`, `n_undamaged`, `false_positives`, and `false_positive_rate`, with one row for each vegetation class.

**You now have the core workflow:** read → inspect → filter → calculate → group → plot → save. The `valid` table also contains the error columns needed for a later logistic-regression lesson.